# Databricks RAG Portfolio — Technical Documentation Q&A

**Author:** Ravi Amaraweera  
**Stack:** Databricks Vector Search + Foundation Model APIs + LangChain (no Mosaic AI Agent Framework)  
**Data:** Apache Airflow open-source docs (Apache 2.0 licence)  
**Estimated cost:** $5–$15 within the free trial $400 credits  

Run each cell top to bottom. Always run Cell 19 at end of session to stop billing.

## Cell 1 — Create Catalog and Schema

Creates the Unity Catalog `rag_portfolio` and schema `doc_search`. Safe to re-run due to `IF NOT EXISTS`.

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS rag_portfolio;
CREATE SCHEMA IF NOT EXISTS doc_search;

## Cell 2 — Install Dependencies

Installs langchain, text-splitters, requests, beautifulsoup4, databricks-langchain.  
Kernel restarts automatically — that is normal. Start from Cell 3 after restart.

In [0]:
%pip install langchain langchain-text-splitters requests beautifulsoup4 databricks-langchain
dbutils.library.restartPython()

## Cell 3 — Download Apache Airflow Documentation

Fetches 7 pages from the official Airflow docs site. BeautifulSoup strips navigation menus
and keeps only the main content text. Each page stored with source URL, text, and doc_type.

In [0]:
import requests
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter

base_url = "https://airflow.apache.org/docs/apache-airflow/stable/"
pages = [
    "tutorial/index.html",
    "howto/operator/index.html",
    "concepts/dags.html",
    "concepts/tasks.html",
    "concepts/operators.html",
    "concepts/connections.html",
    "best-practices.html",
]

documents = []
for page in pages:
    try:
        resp = requests.get(base_url + page, timeout=30)
        soup = BeautifulSoup(resp.text, "html.parser")
        content_div = soup.find("div", class_="body") or soup.find("main")
        if content_div:
            text = content_div.get_text(separator="\n", strip=True)
            documents.append({"source": f"airflow-docs/{page}", "text": text, "doc_type": "airflow"})
            print(f"Downloaded: {page} ({len(text)} chars)")
        else:
            print(f"No content found: {page}")
    except Exception as e:
        print(f"Skipping {page}: {e}")

print(f"\nTotal pages downloaded: {len(documents)}")

## Cell 4 — Chunk the Documents

Splits each page into 1000-character chunks with 150-character overlap.
Smaller chunks give Vector Search better precision. Each chunk becomes one Delta Table row.

In [0]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = []
chunk_id = 0

for doc in documents:
    splits = splitter.split_text(doc["text"])
    for split in splits:
        chunks.append({
            "id": chunk_id,
            "text": split,
            "source": doc["source"],
            "doc_type": doc["doc_type"],
            "char_count": len(split)
        })
        chunk_id += 1

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"Average chunk size: {sum(c['char_count'] for c in chunks) / len(chunks):.0f} chars")

## Cell 5 — Write Chunks to Delta Table

Saves all chunks as a Delta Table in Unity Catalog. This is the source of truth —
Vector Search syncs from this table automatically when new rows are added.

In [0]:
import pandas as pd

table_name = "rag_portfolio.doc_search.airflow_docs_chunks"

df = spark.createDataFrame(pd.DataFrame(chunks))

(df.write
   .format("delta")
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(table_name))

print(f"Wrote {df.count()} chunks to {table_name}")

## Cell 6 — Data Quality Check

Validates data before embedding. Checks for null/empty chunks and prints statistics.
The assert stops execution if bad data is found — pipeline gate best practice.

In [0]:
table_name = "rag_portfolio.doc_search.airflow_docs_chunks"

stats = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_chunks,
        COUNT(DISTINCT source)                                      AS unique_sources,
        ROUND(AVG(char_count), 0)                                   AS avg_chunk_size,
        MIN(char_count)                                             AS min_chunk_size,
        MAX(char_count)                                             AS max_chunk_size,
        SUM(CASE WHEN text IS NULL OR text = '' THEN 1 ELSE 0 END) AS null_chunks
    FROM {table_name}
""")

display(stats)

null_count = stats.collect()[0]["null_chunks"]
assert null_count == 0, f"Found {null_count} null/empty chunks — clean before embedding!"
print("Data quality check passed — safe to proceed to Vector Search")

## Cell 7 — Create Vector Search Endpoint

Provisions the compute server that handles similarity search queries.
**Free trial limit:** 1 endpoint, 1 unit. Costs $0.28/hr even when idle — delete when done (Cell 19).
Takes 5-10 minutes. Run Cell 8 immediately to monitor.

In [0]:
from databricks.vector_search.client import VectorSearchClient

vs_client = VectorSearchClient()
endpoint_name = "rag-portfolio-endpoint"

try:
    vs_client.create_endpoint(name=endpoint_name, endpoint_type="STANDARD")
    print(f"Creating endpoint '{endpoint_name}'... takes 5-10 minutes")
except Exception as e:
    if "already exists" in str(e).lower():
        print(f"Endpoint '{endpoint_name}' already exists — skipping")
    else:
        raise e

## Cell 8 — Wait for Endpoint to Be Ready

Polls every 30 seconds until the endpoint is ONLINE.
Also visible in: Compute → Vector Search tab in the left sidebar.

In [0]:
import time

while True:
    endpoint = vs_client.get_endpoint(endpoint_name)
    status = endpoint.get("endpoint_status", {}).get("state", "UNKNOWN")
    print(f"Endpoint status: {status}")
    if status == "ONLINE":
        print("Endpoint is ready — proceed to Cell 9")
        break
    elif status in ("FAILED", "ERROR"):
        raise Exception(f"Endpoint failed: {status}")
    time.sleep(30)

## Cell 9 — Create the Delta Sync Vector Index

Core step: reads text from Delta Table, sends each chunk to BGE Large embedding model,
stores 1024-dimension vectors in a searchable index.
TRIGGERED = only re-embeds when you call .sync() — cheaper than CONTINUOUS.
Takes 5-15 minutes. Run Cell 10 to monitor.

In [0]:
index_name   = "rag_portfolio.doc_search.airflow_docs_index"
source_table = "rag_portfolio.doc_search.airflow_docs_chunks"

spark.sql(f"ALTER TABLE {source_table} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print("Change Data Feed enabled on source table")

try:
    vs_client.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=source_table,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="text",
        embedding_model_endpoint_name="databricks-bge-large-en",
        columns_to_sync=["id", "text", "source", "doc_type"]
    )
    print(f"Creating index '{index_name}'... takes 5-15 minutes")
except Exception as e:
    if "already exists" in str(e).lower():
        print("Index already exists — skipping")
    else:
        raise e

## Cell 10 — Wait for Index to Be Ready

Polls until all chunks are embedded. When ready=True every chunk has been converted
to a vector and is searchable.

In [0]:
while True:
    index = vs_client.get_index(
        endpoint_name=endpoint_name,
        index_name=index_name
    )
    index_desc = index.describe()
    status  = index_desc.get("status", {}).get("ready", False)
    message = index_desc.get("status", {}).get("message", "indexing...")
    print(f"Index ready: {status} | {message}")
    if status:
        print("Index ready — all chunks embedded and indexed")
        break
    time.sleep(30)

## Cell 11 — Test Direct Similarity Search

Runs a raw similarity search WITHOUT any LLM — purely finds the 3 most semantically
similar chunks to the query. Validate retrieval here before adding the LLM layer.
Scores closer to 1.0 = more similar to the query.

In [0]:
index = vs_client.get_index(
    endpoint_name=endpoint_name,
    index_name=index_name
)

results = index.similarity_search(
    query_text="How do I create an Airflow DAG with multiple tasks?",
    columns=["text", "source", "doc_type"],
    num_results=3
)

print("=== Similarity Search Results ===\n")
for i, row in enumerate(results.get("result", {}).get("data_array", [])):
    print(f"--- Result {i+1} (score: {row[-1]:.4f}) ---")
    print(f"Source : {row[1]}")
    print(f"Text   : {row[0][:300]}...")
    print()

## Cell 12 — Filtered Similarity Search by Metadata

Same as Cell 11 but restricts results to a specific doc_type.
Becomes powerful in Cell 16+ when multiple documentation sources are in the same index.

In [0]:
filtered_results = index.similarity_search(
    query_text="What are best practices for DAG design?",
    columns=["text", "source", "doc_type"],
    filters={"doc_type": "airflow"},
    num_results=3
)

print("=== Filtered Results (Airflow only) ===\n")
for i, row in enumerate(filtered_results.get("result", {}).get("data_array", [])):
    print(f"--- Result {i+1} (score: {row[-1]:.4f}) ---")
    print(f"Source : {row[1]}")
    print(f"Text   : {row[0][:300]}...")
    print()

## Cell 13 — Build the LangChain Retriever and LLM

Sets up two components:
1. DatabricksVectorSearch — wraps the index as a LangChain retriever (k=4 chunks per query)
2. ChatDatabricks — connects to GPT OSS 20B via Foundation Model APIs (cheapest LLM option: 1.0 DBU/M tokens)

In [0]:
from databricks_langchain import DatabricksVectorSearch, ChatDatabricks
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate

vector_store = DatabricksVectorSearch(
    endpoint=endpoint_name,
    index_name=index_name,
    columns=["source", "doc_type"]
)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

llm = ChatDatabricks(
    endpoint="databricks-gpt-oss-20b",
    temperature=0.1,
    max_tokens=500
)

print("Retriever and LLM configured")

## Cell 14 — Define the RAG Prompt and Build the Chain

The prompt instructs the LLM to answer ONLY from retrieved context — reducing hallucination.
chain_type=stuff means all 4 chunks are put into one prompt (simple and cheap).
return_source_documents=True shows which chunks the LLM used — important for demo.

In [0]:
rag_prompt = PromptTemplate(
    input_variables=["context", "question"],
    template="""You are a helpful data engineering assistant. Answer the question
based ONLY on the provided context. If the context does not contain enough
information to answer, say: I don't have enough context to answer this question.

Context:
{context}

Question: {question}

Answer:"""
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    return_source_documents=True,
    chain_type_kwargs={"prompt": rag_prompt}
)

print("RAG chain is ready")

## Cell 15 — Run Test Queries Through the Full RAG Pipeline

Runs 3 real questions end-to-end: query → retrieve 4 chunks → prompt → LLM → answer + sources.
Screenshot these outputs for your GitHub README — they are the key demo outputs.

In [0]:
test_questions = [
    "How do I set up task dependencies in an Airflow DAG?",
    "What is the difference between a Sensor and an Operator in Airflow?",
    "How do I handle failures and retries in Airflow tasks?",
]

for question in test_questions:
    print(f"{'='*65}")
    print(f"Q: {question}")
    result = qa_chain.invoke({"query": question})
    print(f"\nA: {result['result']}")
    print(f"\nSources used:")
    for doc in result["source_documents"]:
        print(f"  - [{doc.metadata.get('doc_type')}] {doc.metadata.get('source')}")
    print()

## Cell 16 — Add dbt Documentation (Incremental Ingestion Demo)

Appends 4 dbt doc chunks to the existing Delta Table. Shows a living knowledge base
that grows as new documentation is added. In production this would be an Airflow DAG
or Delta Live Tables pipeline triggered on a schedule.

In [0]:
import pandas as pd

dbt_chunks = [
    {"id": 10000, "text": "dbt (data build tool) enables analytics engineers to transform data in their warehouses by writing SQL SELECT statements. dbt handles the T in ELT.", "source": "dbt-docs/introduction", "doc_type": "dbt", "char_count": 155},
    {"id": 10001, "text": "A dbt model is a SQL SELECT statement stored in a .sql file. When dbt runs, it wraps each model in a CREATE TABLE AS or CREATE VIEW AS statement automatically.", "source": "dbt-docs/models", "doc_type": "dbt", "char_count": 162},
    {"id": 10002, "text": "dbt tests are assertions you make about your data. Built-in tests check for null values, unique values, accepted values, and referential integrity between tables.", "source": "dbt-docs/tests", "doc_type": "dbt", "char_count": 163},
    {"id": 10003, "text": "dbt sources allow you to declare the tables your dbt project depends on. Declaring sources enables freshness checks so you can alert when upstream data is delayed.", "source": "dbt-docs/sources", "doc_type": "dbt", "char_count": 165},
]

new_df = spark.createDataFrame(pd.DataFrame(dbt_chunks))
new_df.write.format("delta").mode("append").saveAsTable(
    "rag_portfolio.doc_search.airflow_docs_chunks"
)
print(f"Appended {len(dbt_chunks)} dbt chunks — run Cell 17 to sync into Vector Search")

## Cell 17 — Trigger Index Sync for New dbt Chunks

Tells the index to pick up only the new rows added in Cell 16 (incremental — not full re-embed).
Wait 2-3 minutes after running before running Cell 18.

In [0]:
index = vs_client.get_index(
    endpoint_name=endpoint_name,
    index_name=index_name
)
index.sync()
print("Sync triggered — dbt chunks being embedded and added to the index")
print("Wait 2-3 minutes, then run Cell 18")

## Cell 18 — Verify Cross-Source Search (Airflow + dbt Together)

A question about testing should now retrieve chunks from BOTH Airflow and dbt.
If only Airflow sources appear, wait another 2 minutes for sync to complete and re-run.

In [0]:
result = qa_chain.invoke({
    "query": "How do I test my data transformations and pipelines?"
})
print("Q: How do I test my data transformations and pipelines?")
print(f"\nA: {result['result']}")
print(f"\nSources (should include BOTH airflow AND dbt):")
for doc in result["source_documents"]:
    print(f"  - [{doc.metadata.get('doc_type')}] {doc.metadata.get('source')}")

## Cell 19 — CLEANUP: Delete Resources to Stop Billing

The endpoint charges $0.28/hr even when idle. Run this at the end of every session.
The Delta Table data is NOT deleted — only the compute layer.
You can recreate from Cells 7-10 next time (10-15 min to reprovision).

To run: uncomment both delete lines below, then run the cell.

In [0]:
# UNCOMMENT BOTH LINES BELOW AND RUN WHEN DONE FOR THE DAY

# vs_client.delete_index(endpoint_name=endpoint_name, index_name=index_name)
# print("Index deleted")

# vs_client.delete_endpoint(endpoint_name)
# print("Endpoint deleted — billing stopped")

# After running: Compute -> Vector Search -> confirm endpoint is gone

print("Cleanup is commented out for safety. Uncomment both delete lines when ready.")